In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Research Paper Explainer — From PDF to Plain English

## 1. Project Overview

**Task:** Ingest a research paper (PDF or text), extract its sections, generate beginner-friendly summaries, build a glossary of technical terms, and create a quiz — all using a local LLM.

**Approach:** Section-aware paper processing — parse PDF → identify structure → summarize each section → generate glossary → ELI-beginner mode → quiz generation.

**Stack:**
- `pypdf` — PDF text extraction
- `ChatOllama` + `qwen3.5:9b` — local LLM for all generation tasks
- `LangChain` — prompt templates and message formatting

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Ingest** research papers from PDF or raw text |
| 2 | **Extract sections** (Abstract, Introduction, Methods, Results, etc.) using heuristics |
| 3 | **Summarize** each section with controllable detail level |
| 4 | **Generate a glossary** of technical terms with plain-language definitions |
| 5 | **ELI-beginner mode** — re-explain concepts assuming zero background |
| 6 | **Generate quizzes** from paper content to test comprehension |
| 7 | **Chain prompts** — build multi-step LLM pipelines where one output feeds the next |

## 3. Problem Statement

Research papers are dense, jargon-heavy, and assume domain expertise. A CS undergrad reading a transformer paper encounters terms like "multi-head attention", "layer normalization", and "positional encoding" without explanation. Even experienced researchers struggle when reading outside their subfield.

The goal: take any research paper and produce:
1. A **structured breakdown** by section
2. A **plain-language summary** of each section
3. A **glossary** of technical terms
4. A **beginner-friendly explanation** of the entire paper
5. A **quiz** to test understanding

## 4. Why This Project Matters

- **Accessibility** — makes cutting-edge research available to students, journalists, and practitioners
- **Prompt chaining** — demonstrates multi-step LLM pipelines (extract → summarize → transform)
- **Structured output** — glossary and quiz require the LLM to produce formatted JSON/structured text
- **Real-world demand** — tools like Semantic Scholar, Elicit, and SciSpace exist because this problem is massive

## 5. Pipeline Architecture

```
PDF / Raw Text
      │
      ▼
  [Ingest] ── extract text from PDF pages or accept raw text
      │
      ▼
  [Section Extraction] ── detect headings: Abstract, Introduction, Methods, etc.
      │
      ├──▶ [Summary] ── summarize each section (brief + detailed)
      │
      ├──▶ [Glossary] ── identify & define technical terms
      │
      ├──▶ [ELI-Beginner] ── re-explain for someone with zero background
      │
      └──▶ [Quiz] ── generate questions to test comprehension
```

Each step is a separate LLM call. This is **prompt chaining** — the output of ingestion feeds into summarization, which feeds into glossary generation, etc.

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core pypdf

print("Dependencies: langchain, langchain-ollama, pypdf")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

In [ ]:
import os
import re
import json
import textwrap
import warnings
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from pypdf import PdfReader

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

print("All imports OK")

## 8. Configuration

In [ ]:
LLM_MODEL = "qwen3.5:9b"

# ── LLM temperature by task ───────────────────────────
TEMP_EXTRACT = 0.0    # Section extraction — deterministic
TEMP_SUMMARY = 0.3    # Summaries — slight creativity
TEMP_GLOSSARY = 0.2   # Glossary — mostly factual
TEMP_ELI = 0.5        # ELI-beginner — more conversational
TEMP_QUIZ = 0.4       # Quiz — creative but grounded

print("Configuration loaded")
print(f"  LLM: {LLM_MODEL}")
print(f"  Temperatures: extract={TEMP_EXTRACT}, summary={TEMP_SUMMARY}, "
      f"glossary={TEMP_GLOSSARY}, eli={TEMP_ELI}, quiz={TEMP_QUIZ}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    """Strip thinking tags from LLM output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    """Extract JSON from LLM output (handles code fences)."""
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    if start < 0:
        start = text.find("[")
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.0) -> str:
    """Send a prompt to the LLM and return cleaned text."""
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    resp = llm.invoke(messages)
    return clean(resp.content)


def wrap_print(text: str, width: int = 90):
    """Print text with word wrapping."""
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Paper Ingestion

## 10. PDF Text Extraction

We provide two ingestion paths:
1. **PDF file** — extract text from each page using `pypdf`
2. **Raw text** — accept pasted text directly (fallback for environments without a PDF)

Since not every learner has a PDF handy, we include a **sample paper text** (a condensed version of the famous "Attention Is All You Need" paper) so the notebook is always runnable.

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract text from a PDF file.
    Concatenates all pages with page markers.
    """
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = text.strip()
        if text:
            pages.append(f"[Page {i+1}]\n{text}")
    full_text = "\n\n".join(pages)
    print(f"Extracted {len(reader.pages)} pages, {len(full_text):,} characters")
    return full_text


print("PDF extraction function defined")
print("Usage: text = extract_text_from_pdf('path/to/paper.pdf')")

In [ ]:
# ── Sample paper (condensed "Attention Is All You Need") ──
# Used as fallback so the notebook is always runnable.

SAMPLE_PAPER = """Title: Attention Is All You Need

Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones,
Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin

Abstract:
The dominant sequence transduction models are based on complex recurrent or convolutional
neural networks that include an encoder and a decoder. The best performing models also
connect the encoder and decoder through an attention mechanism. We propose a new simple
network architecture, the Transformer, based solely on attention mechanisms, dispensing
with recurrence and convolutions entirely. Experiments on two machine translation tasks
show these models to be superior in quality while being more parallelizable and requiring
significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014
English-to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model
establishes a new single-model state-of-the-art BLEU score of 41.8 after training for
3.5 days on eight GPUs, a small fraction of the training costs of the best models from
the literature. We show that the Transformer generalizes well to other tasks by applying
it successfully to English constituency parsing both with large and limited training data.

1. Introduction:
Recurrent neural networks, long short-term memory and gated recurrent neural networks in
particular, have been firmly established as state of the art approaches in sequence
modeling and transduction problems such as language modeling and machine translation.
Numerous efforts have since continued to push the boundaries of recurrent language models
and encoder-decoder architectures.

Recurrent models typically factor computation along the symbol positions of the input and
output sequences. Aligning the positions to steps in computation time, they generate a
sequence of hidden states ht, as a function of the previous hidden state ht-1 and the
input for position t. This inherently sequential nature precludes parallelization within
training examples, which becomes critical at longer sequence lengths, as memory
constraints limit batching across examples.

Attention mechanisms have become an integral part of compelling sequence modeling and
transduction models in various tasks, allowing modeling of dependencies without regard to
their distance in the input or output sequences. In all but a few cases, however, such
attention mechanisms are used in conjunction with a recurrent network.

In this work, we propose the Transformer, a model architecture eschewing recurrence and
instead relying entirely on an attention mechanism to draw global dependencies between
input and output. The Transformer allows for significantly more parallelization.

2. Background:
The goal of reducing sequential computation also forms the foundation of the Extended
Neural GPU, ByteNet and ConvS2S, all of which use convolutional neural networks as basic
building block, computing hidden representations in parallel for all input and output
positions. In these models, the number of operations required to relate signals from two
arbitrary input or output positions grows in the distance between positions, linearly for
ConvS2S and logarithmically for ByteNet. This makes it more difficult to learn
dependencies between distant positions. In the Transformer this is reduced to a constant
number of operations, albeit at the cost of reduced effective resolution due to averaging
attention-weighted positions, an effect we counteract with Multi-Head Attention.

Self-attention, sometimes called intra-attention, is an attention mechanism relating
different positions of a single sequence in order to compute a representation of the
sequence. Self-attention has been used successfully in a variety of tasks including
reading comprehension, abstractive summarization, textual entailment and learning
task-independent sentence representations.

3. Model Architecture:
Most competitive neural sequence transduction models have an encoder-decoder structure.
Here, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a
sequence of continuous representations z = (z1, ..., zn). Given z, the decoder then
generates an output sequence (y1, ..., ym) of symbols one element at a time. At each
step the model is auto-regressive, consuming the previously generated symbols as
additional input when generating the next.

The Transformer follows this overall architecture using stacked self-attention and
point-wise, fully connected layers for both the encoder and decoder.

3.1 Encoder and Decoder Stacks:
Encoder: The encoder is composed of a stack of N=6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a
simple, position-wise fully connected feed-forward network. We employ a residual
connection around each of the two sub-layers, followed by layer normalization. That is,
the output of each sub-layer is LayerNorm(x + Sublayer(x)).

Decoder: The decoder is also composed of a stack of N=6 identical layers. In addition to
the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which
performs multi-head attention over the output of the encoder stack. We also modify the
self-attention sub-layer in the decoder stack to prevent positions from attending to
subsequent positions (masking).

3.2 Attention:
An attention function can be described as mapping a query and a set of key-value pairs to
an output, where the query, keys, values, and output are all vectors. The output is
computed as a weighted sum of the values, where the weight assigned to each value is
computed by a compatibility function of the query with the corresponding key.

We call our particular attention "Scaled Dot-Product Attention". The input consists of
queries and keys of dimension dk, and values of dimension dv. We compute the dot products
of the query with all keys, divide each by sqrt(dk), and apply a softmax function to
obtain the weights on the values: Attention(Q,K,V) = softmax(QK^T / sqrt(dk))V.

Multi-head attention allows the model to jointly attend to information from different
representation subspaces at different positions. With a single attention head, averaging
inhibits this. MultiHead(Q,K,V) = Concat(head1,...,headh)W^O where
headi = Attention(QW_i^Q, KW_i^K, VW_i^V). We employ h=8 parallel attention heads.

3.3 Position-wise Feed-Forward Networks:
Each layer contains a fully connected feed-forward network, applied to each position
separately and identically. This consists of two linear transformations with a ReLU
activation in between: FFN(x) = max(0, xW1 + b1)W2 + b2.

3.4 Positional Encoding:
Since our model contains no recurrence and no convolution, in order for the model to make
use of the order of the sequence, we must inject some information about the relative or
absolute position of the tokens in the sequence. To this end, we add "positional
encodings" to the input embeddings at the bottoms of the encoder and decoder stacks. We
use sine and cosine functions of different frequencies:
PE(pos, 2i) = sin(pos / 10000^(2i/dmodel))
PE(pos, 2i+1) = cos(pos / 10000^(2i/dmodel))

4. Why Self-Attention:
We compare self-attention layers to recurrent and convolutional layers on three criteria:
total computational complexity per layer, amount of computation that can be parallelized
(measured by minimum sequential operations), and path length between long-range
dependencies. Self-attention layers connect all positions with a constant O(1) number of
sequential operations, whereas a recurrent layer requires O(n) sequential operations.
Self-attention is faster than recurrent layers when the sequence length n is smaller than
the representation dimensionality d, which is most often the case.

5. Training:
We trained on the WMT 2014 English-German dataset consisting of about 4.5 million
sentence pairs. Sentences were encoded using byte-pair encoding, which has a shared
source-target vocabulary of about 37000 tokens. For English-French, we used the larger
WMT 2014 English-French dataset consisting of 36M sentences and split tokens into a
32000 word-piece vocabulary.

We trained on 8 NVIDIA P100 GPUs. Base models were trained for 100,000 steps (12 hours).
Big models were trained for 300,000 steps (3.5 days). We used the Adam optimizer with
beta1=0.9, beta2=0.98 and epsilon=10^-9. We applied dropout of 0.1 and label smoothing
of 0.1.

6. Results:
On the WMT 2014 English-to-German translation task, the big transformer model outperforms
the best previously reported models including ensembles by more than 2.0 BLEU,
establishing a new state-of-the-art BLEU score of 28.4. On the WMT 2014
English-to-French translation task, our big model achieves a BLEU score of 41.0,
outperforming all of the previously published single models, at less than 1/4 the
training cost of the previous state-of-the-art model.

7. Conclusion:
In this work, we presented the Transformer, the first sequence transduction model based
entirely on attention, replacing the recurrent layers most commonly used in
encoder-decoder architectures with multi-headed self-attention. The Transformer can be
trained significantly faster than architectures based on recurrent or convolutional
layers. We achieved a new state of the art on both the WMT 2014 English-to-German and
WMT 2014 English-to-French translation tasks. We plan to extend the Transformer to
other problems, including image, audio, and video, and to investigate local, restricted
attention mechanisms to efficiently handle large inputs and outputs such as images.
"""

print(f"Sample paper loaded: {len(SAMPLE_PAPER):,} characters")
print(f"Title: Attention Is All You Need")

In [ ]:
# ── Load paper: try PDF first, fall back to sample ────
PDF_PATH = None  # ← Set this to your PDF path, e.g. "paper.pdf"

if PDF_PATH and Path(PDF_PATH).exists():
    paper_text = extract_text_from_pdf(PDF_PATH)
    source = f"PDF: {PDF_PATH}"
else:
    if PDF_PATH:
        print(f"⚠ PDF not found: {PDF_PATH}")
    paper_text = SAMPLE_PAPER
    source = "sample (Attention Is All You Need)"

print(f"\nSource: {source}")
print(f"Length: {len(paper_text):,} characters")
print(f"\nFirst 300 chars:")
print(paper_text[:300])

---

# Part B — Section Extraction

## 11. Detect Paper Sections

Research papers follow a common structure (IMRaD): Introduction, Methods, Results, and Discussion. We extract sections using two complementary approaches:

1. **Heuristic** — regex patterns that match numbered headings and common section names
2. **LLM-assisted** — ask the LLM to identify section boundaries (handles unusual formats)

In [ ]:
# ── Approach 1: Heuristic section detection ───────────

SECTION_PATTERNS = [
    # Numbered sections: "1. Introduction", "3.2 Attention"
    r"^\s*(\d+\.?\d*\.?)\s+([A-Z][A-Za-z\s\-:]+)",
    # Named sections without numbers: "Abstract:", "INTRODUCTION"
    r"^\s*(Abstract|Introduction|Background|Related Work|Methods?|Methodology|"
    r"Model Architecture|Experiments?|Results?|Discussion|Conclusion|References?|"
    r"Acknowledgements?|Appendix|Training|Why Self-Attention)\s*:?",
]


def extract_sections_heuristic(text: str) -> list[dict]:
    """
    Extract sections from paper text using regex heading detection.
    Returns list of {heading, content, level} dicts.
    """
    lines = text.split("\n")
    sections = []
    current_heading = "Preamble"
    current_content = []

    for line in lines:
        is_heading = False
        for pattern in SECTION_PATTERNS:
            match = re.match(pattern, line, re.IGNORECASE)
            if match and len(line.strip()) < 80:  # Headings are short
                # Save previous section
                if current_content:
                    content = "\n".join(current_content).strip()
                    if content:
                        sections.append({
                            "heading": current_heading,
                            "content": content,
                        })
                current_heading = line.strip().rstrip(":")
                current_content = []
                is_heading = True
                break

        if not is_heading:
            current_content.append(line)

    # Don't forget the last section
    if current_content:
        content = "\n".join(current_content).strip()
        if content:
            sections.append({
                "heading": current_heading,
                "content": content,
            })

    return sections


# Run section extraction
sections = extract_sections_heuristic(paper_text)

print(f"Detected {len(sections)} sections:")
print("-" * 60)
for i, sec in enumerate(sections):
    print(f"  {i+1:2d}. {sec['heading']:40s} ({len(sec['content']):,} chars)")

In [ ]:
# ── Approach 2: LLM-assisted section detection ────────
# Use when heuristic fails (unusual headings, non-standard format)

LLM_SECTION_PROMPT = """Analyze this research paper text and identify ALL section headings.
Return a JSON array of the section headings in order of appearance.

Example output: ["Abstract", "1. Introduction", "2. Methods", "3. Results", "4. Conclusion"]

Paper text (first 3000 chars):
{text}

Return ONLY the JSON array of headings:"""

resp = ask(
    LLM_SECTION_PROMPT.format(text=paper_text[:3000]),
    temperature=TEMP_EXTRACT,
)
detected = parse_json(resp)

print("LLM-detected sections:")
if detected and isinstance(detected, list):
    for h in detected:
        print(f"  • {h}")
else:
    print(f"  (raw response: {resp[:200]})")

print(f"\n→ Heuristic found {len(sections)} sections")
print(f"→ Both approaches complement each other for robust extraction")

## 12. Inspect Extracted Sections

Look at the first few paragraphs of each section to verify extraction quality.

In [ ]:
for sec in sections:
    print(f"\n{'='*60}")
    print(f"SECTION: {sec['heading']}")
    print(f"{'='*60}")
    # Show first 400 chars of each section
    preview = sec["content"][:400]
    if len(sec["content"]) > 400:
        preview += "..."
    wrap_print(preview)

---

# Part C — Summarization

## 13. Section-by-Section Summary

We summarize each section independently. This is better than summarizing the entire paper at once because:
- Each section has a distinct purpose (intro vs methods vs results)
- The LLM gets focused context (one section at a time)
- Users can jump to the section they care about

In [ ]:
SUMMARY_SYSTEM = """You summarize sections of research papers clearly and accurately.
Write in plain language. A smart undergraduate should understand your summary.
Keep technical terms but briefly explain what they mean.
Be concise — 2-4 sentences per summary."""

SUMMARY_PROMPT = """Summarize this section of a research paper.

Section: {heading}
Content:
{content}

Summary (2-4 sentences):"""

print("SECTION SUMMARIES")
print("=" * 60)

section_summaries = []
for sec in sections:
    # Skip very short sections (e.g., title/author block)
    if len(sec["content"]) < 50:
        continue

    summary = ask(
        SUMMARY_PROMPT.format(
            heading=sec["heading"],
            content=sec["content"][:2000],  # Cap to avoid exceeding context
        ),
        system=SUMMARY_SYSTEM,
        temperature=TEMP_SUMMARY,
    )

    section_summaries.append({
        "heading": sec["heading"],
        "summary": summary,
    })

    print(f"\n📄 {sec['heading']}")
    wrap_print(summary)
    print()

## 14. Full Paper Summary

Now combine the section summaries into one cohesive overview.

In [ ]:
FULL_SUMMARY_PROMPT = """Below are summaries of each section of a research paper.
Write a single cohesive summary of the ENTIRE paper in 1 paragraph (5-8 sentences).
Cover: what problem it solves, the key idea, the method, and the main results.

Section summaries:
{summaries}

Full paper summary (one paragraph):"""

combined = "\n\n".join(
    f"{s['heading']}: {s['summary']}" for s in section_summaries
)

full_summary = ask(
    FULL_SUMMARY_PROMPT.format(summaries=combined),
    temperature=TEMP_SUMMARY,
)

print("FULL PAPER SUMMARY")
print("=" * 60)
wrap_print(full_summary)

---

# Part D — Glossary Generation

## 15. Extract & Define Technical Terms

Research papers use domain-specific jargon. We ask the LLM to:
1. Identify the key technical terms in the paper
2. Define each term in plain language
3. Give a concrete analogy where helpful

In [ ]:
GLOSSARY_SYSTEM = """You build glossaries for research papers. Your audience is an
intelligent person who is NOT an expert in this field. Definitions should be
plain-language, 1-2 sentences each. Add a short analogy if it helps."""

GLOSSARY_PROMPT = """Read this research paper and identify the 12-15 most important
technical terms that a non-expert would need defined.

For each term, provide:
- term: the technical term
- definition: plain-language definition (1-2 sentences)
- analogy: a simple analogy (optional, omit if not helpful)

Return as a JSON array:
[{{"term": "...", "definition": "...", "analogy": "..."}}, ...]

Paper text:
{text}

JSON glossary:"""

glossary_resp = ask(
    GLOSSARY_PROMPT.format(text=paper_text[:4000]),
    system=GLOSSARY_SYSTEM,
    temperature=TEMP_GLOSSARY,
)

glossary = parse_json(glossary_resp)

print("GLOSSARY")
print("=" * 60)

if glossary and isinstance(glossary, list):
    for entry in glossary:
        term = entry.get("term", "?")
        defn = entry.get("definition", "")
        analogy = entry.get("analogy", "")
        print(f"\n  📌 {term}")
        print(f"     {defn}")
        if analogy:
            print(f"     💡 Analogy: {analogy}")
    print(f"\nTotal terms: {len(glossary)}")
else:
    print("(JSON parse failed — raw response below)")
    wrap_print(glossary_resp[:500])

## 16. Contextual Term Lookup

Given a specific term from the paper, explain it in the context of how THIS paper uses it (not just a generic definition).

In [ ]:
LOOKUP_PROMPT = """A reader is studying this research paper and wants to understand
the term "{term}" as used in THIS paper.

Paper context:
{context}

Explain:
1. What "{term}" means in general
2. How this paper specifically uses or defines it
3. Why it matters for the paper's contribution

Use simple language. 3-5 sentences."""

# Look up specific terms
lookup_terms = ["multi-head attention", "positional encoding", "BLEU score"]

print("CONTEXTUAL TERM LOOKUP")
print("=" * 60)

for term in lookup_terms:
    # Find where the term appears in the paper
    context_snippets = []
    for sec in sections:
        if term.lower() in sec["content"].lower():
            context_snippets.append(sec["content"][:500])
    context = "\n...".join(context_snippets[:2]) if context_snippets else paper_text[:1500]

    explanation = ask(
        LOOKUP_PROMPT.format(term=term, context=context),
        temperature=TEMP_GLOSSARY,
    )

    print(f"\n🔍 {term}")
    wrap_print(explanation)
    print()

---

# Part E — Explain Like I'm a Beginner

## 17. ELI-Beginner: Full Paper

Re-explain the entire paper assuming the reader has:
- NO background in machine learning
- NO linear algebra or calculus knowledge
- Basic understanding of computers and programs

In [ ]:
ELI_SYSTEM = """You explain research papers to complete beginners.

Rules:
- Assume the reader has NO background in machine learning, statistics, or advanced math
- Use everyday analogies (kitchens, libraries, schools, sports)
- Avoid jargon — if you must use a technical term, immediately explain it
- Use short sentences and simple words
- Structure your explanation with clear steps: "First... Then... Finally..."
- It's OK to simplify — accuracy can be slightly sacrificed for clarity"""

ELI_PROMPT = """Explain this research paper as if you're talking to a smart
high-school student who has never studied computer science or machine learning.

Cover:
1. What PROBLEM does this paper try to solve? (2-3 sentences)
2. What was the OLD way of solving it? Why was it bad? (2-3 sentences)
3. What is the NEW idea in this paper? (3-5 sentences with analogy)
4. How well does it work? (2-3 sentences)
5. Why should anyone care? (1-2 sentences)

Paper summary:
{summary}

Full paper text (for reference):
{text}

Beginner-friendly explanation:"""

eli_explanation = ask(
    ELI_PROMPT.format(summary=full_summary, text=paper_text[:3000]),
    system=ELI_SYSTEM,
    temperature=TEMP_ELI,
)

print("EXPLAIN LIKE I'M A BEGINNER")
print("=" * 60)
wrap_print(eli_explanation)

## 18. ELI-Beginner: Specific Concepts

Drill into specific hard concepts from the paper.

In [ ]:
CONCEPT_ELI_PROMPT = """Explain the concept of "{concept}" from this research paper
to a complete beginner.

Use:
- A real-world analogy
- No math symbols
- Under 5 sentences

Paper context:
{context}

Simple explanation:"""

hard_concepts = [
    ("self-attention", "Self-attention, sometimes called intra-attention, is an attention "
     "mechanism relating different positions of a single sequence in order to compute a "
     "representation of the sequence."),
    ("encoder-decoder architecture", "Most competitive neural sequence transduction models "
     "have an encoder-decoder structure. The encoder maps an input sequence to a sequence "
     "of continuous representations. The decoder generates an output sequence one element "
     "at a time."),
    ("residual connections and layer normalization", "We employ a residual connection around "
     "each of the two sub-layers, followed by layer normalization. The output of each "
     "sub-layer is LayerNorm(x + Sublayer(x))."),
]

print("CONCEPT-LEVEL BEGINNER EXPLANATIONS")
print("=" * 60)

for concept, context in hard_concepts:
    explanation = ask(
        CONCEPT_ELI_PROMPT.format(concept=concept, context=context),
        system=ELI_SYSTEM,
        temperature=TEMP_ELI,
    )
    print(f"\n🎓 {concept.title()}")
    wrap_print(explanation)
    print()

## 19. Difficulty Levels — Same Content, Different Depths

Show how the same paper section can be explained at three different levels.

In [ ]:
LEVEL_PROMPT = """Explain the main idea of this paper section at {level} level.

{level_instruction}

Section ({heading}):
{content}

Explanation:"""

levels = [
    ("beginner", "Use only everyday words. No technical terms. Use an analogy."),
    ("intermediate", "You can use common CS terms (algorithm, model, training) but "
     "explain domain-specific ones. Brief and clear."),
    ("expert", "Use full technical terminology. Focus on novelty vs prior work. "
     "Be precise and concise."),
]

# Pick a meaty section
target = next((s for s in sections if "attention" in s["heading"].lower()), sections[2])

print(f"SAME SECTION, THREE DIFFICULTY LEVELS")
print(f"Section: {target['heading']}")
print("=" * 60)

for level, instruction in levels:
    explanation = ask(
        LEVEL_PROMPT.format(
            level=level,
            level_instruction=instruction,
            heading=target["heading"],
            content=target["content"][:1500],
        ),
        temperature=TEMP_ELI,
    )
    print(f"\n📊 {level.upper()}")
    wrap_print(explanation)
    print()

---

# Part F — Quiz Generation

## 20. Generate Comprehension Quiz

Create a multiple-choice quiz from the paper content to test reader comprehension.

In [ ]:
QUIZ_SYSTEM = """You create quizzes from research papers. Questions should test
genuine understanding, not trivial recall. Include a mix of:
- Factual questions (what does the paper report?)
- Conceptual questions (why does this approach work?)
- Comparison questions (how does this differ from prior work?)"""

QUIZ_PROMPT = """Generate a 6-question multiple-choice quiz based on this research paper.

For each question provide:
- question: the question text
- options: array of 4 options ["A. ...", "B. ...", "C. ...", "D. ..."]
- correct: the letter of the correct answer ("A", "B", "C", or "D")
- explanation: why the correct answer is right (1-2 sentences)

Return as JSON array:
[{{"question": "...", "options": ["A. ...", ...], "correct": "A", "explanation": "..."}}, ...]

Paper text:
{text}

Section summaries:
{summaries}

JSON quiz (6 questions):"""

summaries_text = "\n".join(f"- {s['heading']}: {s['summary']}" for s in section_summaries)

quiz_resp = ask(
    QUIZ_PROMPT.format(text=paper_text[:4000], summaries=summaries_text),
    system=QUIZ_SYSTEM,
    temperature=TEMP_QUIZ,
)

quiz = parse_json(quiz_resp)

print("COMPREHENSION QUIZ")
print("=" * 60)

if quiz and isinstance(quiz, list):
    for i, q in enumerate(quiz):
        print(f"\nQ{i+1}: {q.get('question', '?')}")
        for opt in q.get("options", []):
            print(f"   {opt}")
        print(f"   ✅ Correct: {q.get('correct', '?')}")
        print(f"   💡 {q.get('explanation', '')}")
    print(f"\nTotal questions: {len(quiz)}")
else:
    print("(JSON parse failed)")
    wrap_print(quiz_resp[:500])

## 21. Interactive Quiz — Test Yourself

In [ ]:
# ── Simulated quiz-taking (auto-answered for notebook runnability) ──
# In a real app, you'd use input() to get user answers.

import random

if quiz and isinstance(quiz, list):
    print("QUIZ — SIMULATED ANSWERS")
    print("=" * 60)

    score = 0
    for i, q in enumerate(quiz):
        correct = q.get("correct", "A").strip().upper()
        # Simulate: 70% chance of correct answer
        choices = ["A", "B", "C", "D"]
        answer = correct if random.random() < 0.7 else random.choice(choices)

        is_correct = answer == correct
        if is_correct:
            score += 1
        icon = "✅" if is_correct else "❌"

        print(f"\n{icon} Q{i+1}: {q.get('question', '')[:80]}")
        print(f"   Your answer: {answer} | Correct: {correct}")
        if not is_correct:
            print(f"   💡 {q.get('explanation', '')}")

    print(f"\nScore: {score}/{len(quiz)} ({100*score/len(quiz):.0f}%)")
else:
    print("No quiz available to take.")

## 22. Generate Open-Ended Discussion Questions

In [ ]:
DISCUSSION_PROMPT = """Based on this research paper, generate 4 open-ended discussion
questions suitable for a reading group or class discussion.

Types:
1. A question about limitations or things the paper doesn't address
2. A question connecting this work to real-world applications
3. A question about potential future work or extensions
4. A critical thinking question about the methodology or evaluation

Paper summary:
{summary}

Return each question on a new line, numbered 1-4:"""

discussion = ask(
    DISCUSSION_PROMPT.format(summary=full_summary),
    temperature=TEMP_QUIZ,
)

print("DISCUSSION QUESTIONS")
print("=" * 60)
wrap_print(discussion)

---

# Part G — Complete Explainer Report

## 23. Generate Full Explainer Document

Combine everything into one structured output — this is the final deliverable.

In [ ]:
print("╔" + "═" * 58 + "╗")
print("║  RESEARCH PAPER EXPLAINER — COMPLETE REPORT              ║")
print("╚" + "═" * 58 + "╝")

# ── Title
title_line = paper_text.split("\n")[0].replace("Title: ", "").strip()
print(f"\n📄 Paper: {title_line}")

# ── One-paragraph summary
print(f"\n{'─'*60}")
print("📝 SUMMARY")
print(f"{'─'*60}")
wrap_print(full_summary)

# ── Section breakdown
print(f"\n{'─'*60}")
print("📑 SECTION-BY-SECTION")
print(f"{'─'*60}")
for s in section_summaries:
    print(f"\n  ▸ {s['heading']}")
    wrap_print("    " + s["summary"])

# ── Glossary
print(f"\n{'─'*60}")
print("📖 GLOSSARY")
print(f"{'─'*60}")
if glossary and isinstance(glossary, list):
    for g in glossary:
        print(f"  • {g.get('term', '?')}: {g.get('definition', '')}")

# ── Beginner explanation
print(f"\n{'─'*60}")
print("🎓 EXPLAIN LIKE I'M A BEGINNER")
print(f"{'─'*60}")
wrap_print(eli_explanation)

# ── Quiz preview
print(f"\n{'─'*60}")
print("❓ QUIZ (first 3 questions)")
print(f"{'─'*60}")
if quiz and isinstance(quiz, list):
    for i, q in enumerate(quiz[:3]):
        print(f"  Q{i+1}: {q.get('question', '?')}")

print(f"\n{'═'*60}")
print(f"Report complete. Sections: {len(section_summaries)} | "
      f"Glossary terms: {len(glossary) if glossary else 0} | "
      f"Quiz questions: {len(quiz) if quiz else 0}")

---

## 24. Error Analysis — Where the Pipeline Struggles

In [ ]:
# ── Stress test: feed the LLM a very short section ────
SHORT_TEXT = "We use dropout of 0.1."

print("EDGE CASE 1: Very short section")
print(f"Input: '{SHORT_TEXT}'")
short_summary = ask(
    SUMMARY_PROMPT.format(heading="Training Details", content=SHORT_TEXT),
    system=SUMMARY_SYSTEM,
    temperature=TEMP_SUMMARY,
)
print(f"Summary: {short_summary}")
print()

# ── Stress test: ask for glossary on non-technical text ──
SIMPLE_TEXT = "The cat sat on the mat. It was a nice day."

print("EDGE CASE 2: Non-technical text")
simple_gloss = ask(
    GLOSSARY_PROMPT.format(text=SIMPLE_TEXT),
    system=GLOSSARY_SYSTEM,
    temperature=TEMP_GLOSSARY,
)
print(f"Glossary response: {simple_gloss[:200]}")
print()

# ── Stress test: math-heavy content ───────────────────
MATH_TEXT = "Attention(Q,K,V) = softmax(QK^T / sqrt(dk))V where dk = 64."

print("EDGE CASE 3: Math-heavy content → beginner explanation")
math_eli = ask(
    CONCEPT_ELI_PROMPT.format(concept="the attention formula", context=MATH_TEXT),
    system=ELI_SYSTEM,
    temperature=TEMP_ELI,
)
print(f"Beginner explanation: {math_eli}")

## 25. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **Feeding the entire paper as one prompt** | Exceeds context window; LLM ignores middle sections | Process section by section, then combine |
| **No section detection** | Treating the paper as flat text loses structure | Use heading detection (regex + LLM) to split |
| **Generic glossary** | Defining terms out of context ("attention = focus") | Define terms as used IN THIS PAPER |
| **Oversimplified ELI-beginner** | "It's a smart computer" — not useful | Keep analogies specific: what does it DO, step by step |
| **No grounding for quiz** | Quiz questions about things not in the paper | Generate quiz ONLY from paper content |
| **Ignoring extraction errors** | PDF tables, figures, and equations extract as garbage | Detect & skip non-text content; warn the user |
| **Single difficulty level** | Experts find beginner explanations patronizing; beginners find expert ones useless | Offer multiple levels (beginner/intermediate/expert) |
| **Not validating JSON output** | LLM sometimes returns invalid JSON | Always use a parse_json helper with fallback |

## 26. Production Improvement Ideas

1. **Figure & table extraction** — Use a vision model (GPT-4V, LLaVA) to describe figures and tables
2. **Citation graph** — Extract references and show which papers are most cited in the text
3. **Comparative mode** — Upload two papers, get a comparison of their approaches and results
4. **Flashcard generation** — Export glossary terms as Anki flashcards for spaced repetition
5. **Audio summary** — Generate a 5-minute podcast-style explanation using TTS
6. **Streaming UI** — Show summaries as they generate (section by section) in a web UI
7. **RAG over paper** — Chunk the paper into a vector store so users can ask follow-up questions
8. **Multi-paper reading list** — Process 5 papers on a topic and generate a literature review

## 27. Exercises

### Exercise 1: Use Your Own Paper

In [ ]:
# ── Exercise 1: Replace with your own PDF ─────────────
# 1. Set PDF_PATH to your paper
# 2. Re-run from cell 11 onward

# PDF_PATH = "my_paper.pdf"
# paper_text = extract_text_from_pdf(PDF_PATH)
# sections = extract_sections_heuristic(paper_text)
# ... then re-run summarization, glossary, ELI, quiz cells

print("Exercise 1: Set PDF_PATH to your paper and re-run the pipeline.")

### Exercise 2: Add a "Key Contributions" Extractor

In [ ]:
# ── Exercise 2: Extract key contributions ─────────────

CONTRIBUTIONS_PROMPT = """Read this research paper and list its KEY CONTRIBUTIONS.
Each contribution should be one sentence stating what the paper accomplished.
Most papers have 2-4 key contributions.

Paper text:
{text}

Return as a numbered list:"""

contributions = ask(
    CONTRIBUTIONS_PROMPT.format(text=paper_text[:3000]),
    temperature=TEMP_EXTRACT,
)

print("KEY CONTRIBUTIONS")
print("=" * 60)
wrap_print(contributions)

### Exercise 3: Generate a Twitter Thread

In [ ]:
# ── Exercise 3: Paper → Twitter thread ────────────────

THREAD_PROMPT = """Convert this research paper summary into a Twitter/X thread of 5 tweets.

Rules:
- First tweet: hook that grabs attention + paper title
- Tweets 2-4: key findings, one per tweet
- Last tweet: why it matters + call to action
- Each tweet ≤ 280 characters
- Use emoji sparingly (1-2 per tweet)

Paper summary:
{summary}

Thread (numbered 1-5):"""

thread = ask(
    THREAD_PROMPT.format(summary=full_summary),
    temperature=0.6,
)

print("TWITTER THREAD")
print("=" * 60)
wrap_print(thread)

### Mini Challenge

1. **Automatic difficulty detection:** Analyze the paper's vocabulary complexity (avg word length, jargon density) and auto-select the right explanation level. Papers with > 30% jargon should default to beginner mode.

2. **Multi-paper comparison:** Accept 2-3 papers, extract their methods sections, and generate a comparison table showing: approach, dataset, metric, result. Which paper has the best methodology?

3. **Adaptive quiz difficulty:** Start with easy questions. If the user gets 3/3 correct, switch to hard questions. If they miss 2, switch to easier ones. Implement this as a state machine.

4. **Citation context extraction:** For each cited paper [N], extract the sentence(s) where it's cited and explain WHY the authors cite that work. This helps understand the paper's intellectual lineage.

## 28. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

print(f"\nPaper: {title_line}")
print(f"Source: {source}")
print(f"Paper length: {len(paper_text):,} chars")
print(f"Sections detected: {len(sections)}")
print(f"Sections summarized: {len(section_summaries)}")
print(f"Glossary terms: {len(glossary) if glossary and isinstance(glossary, list) else 0}")
print(f"Quiz questions: {len(quiz) if quiz and isinstance(quiz, list) else 0}")

print(f"\nPipeline stages completed:")
print(f"  ✅ PDF/text ingestion")
print(f"  ✅ Section extraction (heuristic + LLM)")
print(f"  ✅ Section summaries + full summary")
print(f"  ✅ Glossary generation (with analogies)")
print(f"  ✅ ELI-beginner (full paper + concepts + 3 difficulty levels)")
print(f"  ✅ Quiz generation (MCQ + discussion questions)")
print(f"  ✅ Complete explainer report")

## 29. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Section-level processing** beats whole-paper prompts — it stays within context limits and preserves structure |
| 2 | **Prompt chaining** (extract → summarize → transform) is the core pattern for document understanding |
| 3 | **Temperature control matters** — use 0.0 for extraction, 0.3 for summaries, 0.5+ for creative rewrites |
| 4 | **Structured output (JSON)** is essential for glossary and quiz — always have a parse_json fallback |
| 5 | **Multiple difficulty levels** serve different audiences from the same source material |
| 6 | **Heuristic + LLM** section detection is more robust than either alone |
| 7 | **Edge cases** (short text, math-heavy content, non-technical input) reveal where LLM pipelines break |
| 8 | **The full report** combining summary + glossary + ELI + quiz is more useful than any single component |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*